# Run experiment on fMRI datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import sys

print(f"Python: {sys.version.split()[0]}")
print(f"numpy: {np.__version__}, pandas: {pd.__version__}")
np.set_printoptions(precision=3, suppress=True)


from src.data_preprocessing import preprocess_data
from src.plotting import plot_heatmap
from src.causal_matrix_evaluation import evaluate_causal_matrices
from src.matrix_utils import read_matrices_from_csv, save_matrices, get_summary_matrix
from src.run_causal_discovery import run_pcmci, run_vcdf_pcmci, run_varlingam, run_vcdf_varlingam, run_tcdf, run_dynotears

In [ ]:
def run_experiments(dataset_indices, methods=['pcmci', 'vcdf_pcmci', 'varlingam', 'vcdf_varlingam', 'tcdf', 'dynotears']):
    """
    Run causal discovery experiments on fMRI datasets.
    
    Args:
        dataset_indices: List of dataset indices to process
        methods: List of methods to run
    """
    results = {method: [] for method in methods}
    
    for index in dataset_indices:
        print(f"Processing dataset {index}")
        
        # Load ground truths
        ground_truths_path = f'data/real/fMRI/ground_truths/sim{index}_gt_processed_adj.csv'
        ground_truths = read_matrices_from_csv(ground_truths_path)
        
        if ground_truths is None:
            print(f"Skipping dataset {index} due to missing ground truths")
            continue
            
        # Get summary ground truth matrix
        ground_truth_summary = get_summary_matrix(ground_truths)
        
        for method in methods:
            # Load data
            data = pd.read_csv(f'data/real/fMRI/returns/timeseries{index}.csv')
            columns = data.columns.tolist()
            
            # Remove timestamp column if present
            for time_col in ['Date', 'timestamp']:
                if time_col in columns:
                    data = data.drop([time_col], axis=1)
                    columns.remove(time_col)
            
            data = data.values

            # Preprocess data
            data = preprocess_data(data, columns)

            # Run causal discovery method
            start_time = time.time()
            
            if method == 'pcmci':
                adjacency_matrices = run_pcmci(data)
            elif method == 'vcdf_pcmci':
                adjacency_matrices = run_vcdf_pcmci(data)
            elif method == 'varlingam':
                adjacency_matrices = run_varlingam(data)
            elif method == 'vcdf_varlingam':
                adjacency_matrices = run_vcdf_varlingam(data)
            elif method == 'tcdf':
                adjacency_matrices = run_tcdf(data)
            elif method == 'dynotears':
                adjacency_matrices = run_dynotears(data)
            else:
                raise ValueError(f"Unknown method: {method}")
                
            runtime = round(time.time() - start_time, 4)

            # Trim adjacency matrices if needed
            if len(adjacency_matrices) > len(ground_truths):
                adjacency_matrices = adjacency_matrices[:len(ground_truths)]

            # Get summary matrix from method results
            method_summary = get_summary_matrix(adjacency_matrices)

            # Save results
            output_path = f'results/real/fMRI/timeseries{index}/adj_matrices_{method}.csv'
            summary_path = f'results/real/fMRI/timeseries{index}/sum_adj_matrix_{method}.csv'
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            save_matrices(adjacency_matrices, output_path)
            save_matrices([method_summary], summary_path)

            # Evaluate both full and summary matrices
            full_evaluation = evaluate_causal_matrices(ground_truths, adjacency_matrices)
            summary_evaluation = evaluate_causal_matrices([ground_truth_summary], [method_summary])

            # Store results
            result = {
                'dataset': index,
                'Full_SHD': full_evaluation['shd'],
                'Full_F1': full_evaluation['f1'],
                'Full_F1_sign': full_evaluation['f1_sign'],
                'Full_Frobenius': full_evaluation['fro'],
                'Summary_SHD': summary_evaluation['shd'],
                'Summary_F1': summary_evaluation['f1'],
                'Summary_F1_sign': summary_evaluation['f1_sign'],
                'Summary_Frobenius': summary_evaluation['fro'],
                'runtime': runtime
            }
            results[method].append(result)

    # Save summary results for each method
    for method in methods:
        df_results = pd.DataFrame(results[method])
        
        # Calculate statistics for numeric columns
        numeric_cols = df_results.select_dtypes(include=[np.number]).columns
        numeric_cols = [col for col in numeric_cols if col not in ['dataset']]
        avg_result = df_results[numeric_cols].mean()
        std_result = df_results[numeric_cols].std()

        # Create summary row
        summary = {
            'dataset': 'Overall Average',
            **{col: (f"{avg_result[col]:.4f} ± {std_result[col]:.4f}" 
                    if col != 'runtime' else f"{avg_result[col]:.4f}")
               for col in numeric_cols}
        }
        
        # Add summary to results
        df_results = pd.concat([df_results, pd.DataFrame([summary])], ignore_index=True)
        
        # Save to CSV
        output_path = f'results/real/fMRI/performance_{method}.csv'
        df_results.to_csv(output_path, index=False)

In [ ]:
# Define datasets to process (excluding dataset 4)
dataset_indices = [i for i in range(1, 29) if i != 4]

# Define methods to run
methods_to_run = ['pcmci', 'vcdf_pcmci', 'varlingam', 'vcdf_varlingam', 'tcdf', 'dynotears']

# Run experiments
run_experiments(dataset_indices, methods=methods_to_run)